## 🎯 Learning Objectives
* Design and implement a comprehensive evaluation pipeline for an LLM application.
* Utilize modern evaluation frameworks (e.g., Ragas) to assess LLM application performance.
* Define and generate synthetic or mock datasets suitable for LLM evaluation.
* Integrate evaluation results with observability and tracking tools (e.g., Langfuse) for continuous improvement.
* Analyze and interpret evaluation metrics to identify areas for LLM application enhancement.


## Exercise: Building a Full LLM Evaluation Pipeline

### Task Description

In this exercise, you will design and implement a complete evaluation pipeline for a simplified Retrieval-Augmented Generation (RAG) application. This RAG application will answer questions based on a small, predefined set of technical documentation snippets. Your goal is to ensure the application provides accurate, relevant, and well-grounded answers.

**Scenario:**
You are an AI engineer working on a new internal knowledge base Q&A system for your company's `AgenticLabs.ng` documentation. The system uses a RAG architecture to fetch relevant documentation snippets and generate answers. Before deploying this system, you need to establish a robust evaluation pipeline to measure its performance and identify potential issues.

### Requirements

1.  **Mock Data Generation:** Create a small, synthetic dataset for evaluation. This dataset should include:
    *   `question`: The user's query.
    *   `ground_truth`: The expected correct answer to the question.
    *   `contexts`: A list of relevant documentation snippets that *should* be retrieved to answer the question.
    *   `documents`: A larger pool of documents from which the RAG system will retrieve.

2.  **Simplified RAG Application:** Implement a basic RAG application that:
    *   Takes a `question` as input.
    *   Performs a simple retrieval step (e.g., keyword search or mock embedding search) over the `documents` to get relevant `contexts`.
    *   Uses a mock LLM (or a real one if you have API access, e.g., OpenAI's `gpt-3.5-turbo`) to generate an `answer` based on the `question` and retrieved `contexts`.
    *   Returns the generated `answer` and the `retrieved_contexts`.

3.  **Evaluation Metrics:** Select and implement at least three relevant evaluation metrics for RAG systems. We recommend using the `ragas` library for this, focusing on metrics like:
    *   `faithfulness`: Measures if the generated answer is grounded in the retrieved contexts.
    *   `answer_relevance`: Measures how relevant the generated answer is to the question.
    *   `context_relevance`: Measures how relevant the retrieved contexts are to the question.

4.  **Evaluation Execution:** Run your RAG application against the generated dataset and collect the necessary outputs (`question`, `answer`, `retrieved_contexts`, `ground_truth`) for evaluation.

5.  **Results Reporting:** Display the aggregated evaluation scores (e.g., average scores for each metric) in a clear and readable format. Optionally, display individual scores for each data point.

6.  **Observability & Tracking (Bonus/Recommended for 2026):** Integrate with an LLM observability tool like Langfuse (or MLflow) to log the RAG traces and evaluation results. This will help in tracking experiments and debugging.

### Evaluation Criteria

Your solution will be evaluated based on the following:

*   **Correctness:** Does the pipeline run without errors and produce meaningful results?
*   **Completeness:** Are all required components (data, RAG app, metrics, reporting) implemented?
*   **Clarity & Readability:** Is the code well-structured, commented, and easy to understand?
*   **Modern Tooling:** Effective use of libraries like `ragas` and consideration for observability tools.
*   **Reproducibility:** Can the evaluation be easily re-run and yield consistent results?


In [ ]:
# Setup Code: Install libraries, mock data, and basic configurations

# Install necessary libraries (uncomment and run if not already installed)
# !pip install -qU langchain openai ragas datasets pandas langfuse

import os
import json
import pandas as pd
from typing import List, Dict, Any

# Mock LLM for demonstration purposes if you don't have an OpenAI API key
# In a real scenario, you would use an actual LLM client (e.g., OpenAI, Anthropic, Cohere)
class MockLLM:
    def __init__(self, delay_seconds=0.1):
        self.delay_seconds = delay_seconds

    def generate(self, prompt: str) -> str:
        # Simulate LLM thinking and response generation
        import time
        time.sleep(self.delay_seconds)

        if "What is AgenticLabs.ng?" in prompt:
            return "AgenticLabs.ng is a leading platform specializing in Agentic AI and Automation Tools, providing cutting-edge solutions for AI engineers and DevOps specialists."
        elif "What is LLMOps?" in prompt:
            return "LLMOps refers to the practices and tools for managing the lifecycle of Large Language Models, from development and evaluation to deployment and monitoring in production."
        elif "Who is the target audience for OPS-01?" in prompt:
            return "The target audience for OPS-01 is AI engineers and DevOps specialists looking to master LLMOps."
        elif "What are the prerequisites for OPS-01?" in prompt:
            return "The prerequisite for OPS-01 is LLM-01, which covers foundational knowledge in Large Language Models."
        elif "What is the current lesson about?" in prompt:
            return "The current lesson, OPS01-L08, is an exercise focused on setting up a full evaluation pipeline for an LLM application."
        else:
            return "I'm sorry, I can only answer questions related to AgenticLabs.ng and LLMOps based on my internal knowledge. Please provide more context if you have a specific query."

# --- Mock Data Generation ---
# This simulates our internal documentation and a small evaluation dataset.

documents = [
    "AgenticLabs.ng is a leading platform specializing in Agentic AI and Automation Tools.",
    "Our mission is to empower AI engineers and DevOps specialists with cutting-edge solutions.",
    "Course OPS-01: Mastering LLMOps: From Build to Deployment, covers automated evaluation pipelines, CI evals, model serving optimizations, and production observability stack.",
    "The target audience for OPS-01 is AI engineers and DevOps specialists.",
    "Prerequisites for OPS-01 include LLM-01, which focuses on foundational LLM concepts.",
    "Lesson OPS01-L08 is an exercise on setting up a full evaluation pipeline for an LLM application.",
    "LLMOps involves managing the entire lifecycle of LLMs, including evaluation, deployment, and monitoring.",
    "Automated evaluation pipelines are crucial for ensuring the quality and reliability of LLM applications in production."
]

# Evaluation dataset format: question, ground_truth, and contexts (expected relevant contexts)
# In a real scenario, ground_truth and contexts might be manually curated or generated.

eval_dataset_raw = [
    {
        "question": "What is AgenticLabs.ng?",
        "ground_truth": "AgenticLabs.ng is a platform focused on Agentic AI and Automation Tools, designed for AI engineers and DevOps specialists.",
        "contexts": [
            "AgenticLabs.ng is a leading platform specializing in Agentic AI and Automation Tools.",
            "Our mission is to empower AI engineers and DevOps specialists with cutting-edge solutions."
        ]
    },
    {
        "question": "What does Course OPS-01 cover?",
        "ground_truth": "Course OPS-01, 'Mastering LLMOps: From Build to Deployment', covers automated evaluation pipelines, CI evals, model serving optimizations, and production observability.",
        "contexts": [
            "Course OPS-01: Mastering LLMOps: From Build to Deployment, covers automated evaluation pipelines, CI evals, model serving optimizations, and production observability stack."
        ]
    },
    {
        "question": "Who is the target audience for the Mastering LLMOps course?",
        "ground_truth": "The Mastering LLMOps course (OPS-01) is designed for AI engineers and DevOps specialists.",
        "contexts": [
            "The target audience for OPS-01 is AI engineers and DevOps specialists."
        ]
    },
    {
        "question": "What are the prerequisites for OPS-01?",
        "ground_truth": "The prerequisite for OPS-01 is LLM-01, which covers foundational LLM concepts.",
        "contexts": [
            "Prerequisites for OPS-01 include LLM-01, which focuses on foundational LLM concepts."
        ]
    },
    {
        "question": "Why are automated evaluation pipelines important in LLMOps?",
        "ground_truth": "Automated evaluation pipelines are critical in LLMOps for ensuring the quality, reliability, and continuous improvement of LLM applications in production environments.",
        "contexts": [
            "Automated evaluation pipelines are crucial for ensuring the quality and reliability of LLM applications in production.",
            "LLMOps involves managing the entire lifecycle of LLMs, including evaluation, deployment, and monitoring."
        ]
    }
]

print(f"Loaded {len(documents)} mock documents.")
print(f"Loaded {len(eval_dataset_raw)} evaluation samples.")

# --- LLM Configuration ---
# Set your OpenAI API key if you want to use a real LLM.
# Otherwise, the MockLLM will be used.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# --- Langfuse Configuration (Optional but Recommended) ---
# Set your Langfuse API keys for observability.
# os.environ["LANGFUSE_PUBLIC_KEY"] = "YOUR_LANGFUSE_PUBLIC_KEY"
# os.environ["LANGFUSE_SECRET_KEY"] = "YOUR_LANGFUSE_SECRET_KEY"
# os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com" # Or your self-hosted instance

# Initialize the LLM. Use MockLLM if no OpenAI key is set.
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    print("Using OpenAI ChatOpenAI LLM.")
else:
    llm = MockLLM()
    print("Using MockLLM. For real LLM, set OPENAI_API_KEY environment variable.")

# Initialize Langfuse if keys are set
langfuse_enabled = False
if os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"):
    from langfuse import Langfuse
    langfuse = Langfuse()
    langfuse_enabled = True
    print("Langfuse tracing enabled.")
else:
    print("Langfuse tracing disabled. Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY for observability.")

print("Setup complete. Proceed to implement your RAG application and evaluation pipeline.")


## Your Turn: Implement the LLM Evaluation Pipeline

Now it's your turn to build the RAG application and the full evaluation pipeline. Follow the requirements outlined in the task description.

In the cell below, implement the following:

1.  **RAG Application (`rag_application` function):**
    *   Take a `question` as input.
    *   Implement a simple retriever that finds relevant `contexts` from the `documents` list provided in the setup code. A basic keyword search is sufficient for this exercise.
    *   Construct a prompt for the LLM using the `question` and `retrieved_contexts`.
    *   Call the `llm` (either `MockLLM` or `ChatOpenAI`) to generate an `answer`.
    *   Return the `answer` and the `retrieved_contexts`.
    *   *(Optional for Langfuse)* If Langfuse is enabled, wrap your RAG components (retrieval, generation) with Langfuse spans to trace the execution.

2.  **Evaluation Loop:**
    *   Iterate through the `eval_dataset_raw`.
    *   For each sample, call your `rag_application` to get the `answer` and `retrieved_contexts`.
    *   Store the results in a format suitable for `ragas` evaluation (e.g., a list of dictionaries with `question`, `answer`, `contexts`, `ground_truth`).

3.  **Ragas Evaluation:**
    *   Convert your collected results into a `datasets.Dataset` object.
    *   Initialize `ragas` metrics (e.g., `Faithfulness`, `AnswerRelevance`, `ContextRelevance`).
    *   Run the `ragas` evaluation.

4.  **Reporting:**
    *   Print the aggregated `ragas` scores.
    *   *(Optional)* Display a `pandas` DataFrame showing individual scores per sample.

5.  **Langfuse Integration (if enabled):**
    *   Log the overall evaluation results to Langfuse as a `score` or `metric` for the trace.
    *   Ensure each RAG call is traced with its inputs, outputs, and potentially intermediate steps.

Remember to add comments to your code to explain your design choices and logic.


In [ ]:
# Reference Solution: Full LLM Evaluation Pipeline

import os
import json
import pandas as pd
from typing import List, Dict, Any
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevance, context_relevance

# Re-import setup variables to ensure they are available in this cell
# In a real notebook, these would persist from the previous cell.
# For robustness in a standalone solution, we might re-define or ensure import.

# Mock LLM (re-defined for self-contained solution, assuming previous cell might not be run)
class MockLLM:
    def __init__(self, delay_seconds=0.1):
        self.delay_seconds = delay_seconds

    def generate(self, prompt: str) -> str:
        import time
        time.sleep(self.delay_seconds)
        if "What is AgenticLabs.ng?" in prompt:
            return "AgenticLabs.ng is a leading platform specializing in Agentic AI and Automation Tools, providing cutting-edge solutions for AI engineers and DevOps specialists."
        elif "What does Course OPS-01 cover?" in prompt:
            return "Course OPS-01, 'Mastering LLMOps: From Build to Deployment', covers automated evaluation pipelines, CI evals, model serving optimizations, and production observability stack."
        elif "Who is the target audience for the Mastering LLMOps course?" in prompt:
            return "The target audience for OPS-01 is AI engineers and DevOps specialists looking to master LLMOps."
        elif "What are the prerequisites for OPS-01?" in prompt:
            return "The prerequisite for OPS-01 is LLM-01, which covers foundational knowledge in Large Language Models."
        elif "Why are automated evaluation pipelines important in LLMOps?" in prompt:
            return "Automated evaluation pipelines are crucial for ensuring the quality and reliability of LLM applications in production."
        else:
            return "I'm sorry, I can only answer questions related to AgenticLabs.ng and LLMOps based on my internal knowledge. Please provide more context if you have a specific query."

# Documents and evaluation dataset (re-defined for self-contained solution)
documents = [
    "AgenticLabs.ng is a leading platform specializing in Agentic AI and Automation Tools.",
    "Our mission is to empower AI engineers and DevOps specialists with cutting-edge solutions.",
    "Course OPS-01: Mastering LLMOps: From Build to Deployment, covers automated evaluation pipelines, CI evals, model serving optimizations, and production observability stack.",
    "The target audience for OPS-01 is AI engineers and DevOps specialists.",
    "Prerequisites for OPS-01 include LLM-01, which focuses on foundational LLM concepts.",
    "Lesson OPS01-L08 is an exercise on setting up a full evaluation pipeline for an LLM application.",
    "LLMOps involves managing the entire lifecycle of LLMs, including evaluation, deployment, and monitoring.",
    "Automated evaluation pipelines are crucial for ensuring the quality and reliability of LLM applications in production."
]

eval_dataset_raw = [
    {
        "question": "What is AgenticLabs.ng?",
        "ground_truth": "AgenticLabs.ng is a platform focused on Agentic AI and Automation Tools, designed for AI engineers and DevOps specialists.",
        "contexts": [
            "AgenticLabs.ng is a leading platform specializing in Agentic AI and Automation Tools.",
            "Our mission is to empower AI engineers and DevOps specialists with cutting-edge solutions."
        ]
    },
    {
        "question": "What does Course OPS-01 cover?",
        "ground_truth": "Course OPS-01, 'Mastering LLMOps: From Build to Deployment', covers automated evaluation pipelines, CI evals, model serving optimizations, and production observability stack.",
        "contexts": [
            "Course OPS-01: Mastering LLMOps: From Build to Deployment, covers automated evaluation pipelines, CI evals, model serving optimizations, and production observability stack."
        ]
    },
    {
        "question": "Who is the target audience for the Mastering LLMOps course?",
        "ground_truth": "The Mastering LLMOps course (OPS-01) is designed for AI engineers and DevOps specialists.",
        "contexts": [
            "The target audience for OPS-01 is AI engineers and DevOps specialists."
        ]
    },
    {
        "question": "What are the prerequisites for OPS-01?",
        "ground_truth": "The prerequisite for OPS-01 is LLM-01, which covers foundational LLM concepts.",
        "contexts": [
            "Prerequisites for OPS-01 include LLM-01, which focuses on foundational LLM concepts."
        ]
    },
    {
        "question": "Why are automated evaluation pipelines important in LLMOps?",
        "ground_truth": "Automated evaluation pipelines are critical in LLMOps for ensuring the quality, reliability, and continuous improvement of LLM applications in production environments.",
        "contexts": [
            "Automated evaluation pipelines are crucial for ensuring the quality and reliability of LLM applications in production.",
            "LLMOps involves managing the entire lifecycle of LLMs, including evaluation, deployment, and monitoring."
        ]
    }
]

# Initialize the LLM. Use MockLLM if no OpenAI key is set.
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    print("Using OpenAI ChatOpenAI LLM.")
else:
    llm = MockLLM()
    print("Using MockLLM. For real LLM, set OPENAI_API_KEY environment variable.")

# Initialize Langfuse if keys are set
langfuse_enabled = False
if os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"):
    from langfuse import Langfuse
    from langfuse.model import InitialGeneration, InitialSpan
    langfuse = Langfuse()
    langfuse_enabled = True
    print("Langfuse tracing enabled.")
else:
    print("Langfuse tracing disabled. Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY for observability.")


# --- 1. RAG Application Implementation ---

def retrieve_contexts(query: str, docs: List[str], top_k: int = 2) -> List[str]:
    """A simple keyword-based retriever."""
    # In a real RAG, this would involve embeddings, vector databases, etc.
    # For this exercise, we'll do a basic keyword match.
    retrieved = []
    query_lower = query.lower()
    for doc in docs:
        if any(keyword in doc.lower() for keyword in query_lower.split()):
            retrieved.append(doc)
    # Simple heuristic: return documents that contain any word from the query
    # Or, for a more deterministic mock, we could just return the 'expected' contexts
    # For this exercise, let's make it slightly more dynamic but still simple.
    if not retrieved:
        # Fallback if no keywords match, just return a couple of general docs
        return docs[:top_k]
    return list(set(retrieved))[:top_k] # Remove duplicates and limit

def rag_application(question: str, docs: List[str], trace_name: str = "rag_pipeline") -> Dict[str, Any]:
    """Simplified RAG application that retrieves contexts and generates an answer."""
    retrieved_contexts = []
    generated_answer = ""
    
    # Start a Langfuse trace if enabled
    trace = None
    if langfuse_enabled:
        trace = langfuse.trace(name=trace_name, input=question)

    try:
        # --- Retrieval Step ---
        retrieval_span = None
        if langfuse_enabled and trace:
            retrieval_span = trace.span(InitialSpan(name="retrieval", input=question))
        
        retrieved_contexts = retrieve_contexts(question, docs)
        
        if langfuse_enabled and retrieval_span:
            retrieval_span.end(output=retrieved_contexts)

        # --- Generation Step ---
        prompt_template = (
            "You are a helpful assistant. Answer the following question based only on the provided context.\n\n"
            "Context:\n{context}\n\n"
            "Question: {question}\n\n"
            "Answer:"
        )
        context_str = "\n".join(retrieved_contexts)
        llm_prompt = prompt_template.format(context=context_str, question=question)

        generation_span = None
        if langfuse_enabled and trace:
            generation_span = trace.span(InitialSpan(name="generation", input=llm_prompt))

        if isinstance(llm, MockLLM):
            generated_answer = llm.generate(llm_prompt)
        else:
            # For LangChain LLMs, use invoke
            generated_answer = llm.invoke(llm_prompt).content

        if langfuse_enabled and generation_span:
            generation_span.end(output=generated_answer)

    except Exception as e:
        print(f"Error during RAG application for question '{question}': {e}")
        if langfuse_enabled and trace:
            trace.update(level="ERROR", status_message=str(e))
        generated_answer = "An error occurred while processing your request."
    finally:
        if langfuse_enabled and trace:
            trace.end(output=generated_answer)

    return {
        "question": question,
        "answer": generated_answer,
        "contexts": retrieved_contexts # These are the *actually retrieved* contexts
    }


# --- 2. Evaluation Loop ---
print("\nRunning RAG application for evaluation...")

rag_results = []
for i, sample in enumerate(eval_dataset_raw):
    # Pass the question and the full set of documents to the RAG app
    app_output = rag_application(sample["question"], documents, trace_name=f"rag_eval_sample_{i}")
    
    # Combine app output with ground truth for Ragas dataset format
    rag_results.append({
        "question": app_output["question"],
        "answer": app_output["answer"],
        "contexts": app_output["contexts"],
        "ground_truth": sample["ground_truth"]
    })
    print(f"Processed sample {i+1}/{len(eval_dataset_raw)}")

# --- 3. Ragas Evaluation ---
print("\nStarting Ragas evaluation...")

# Convert results to Hugging Face Dataset format required by Ragas
eval_dataset = Dataset.from_list(rag_results)

# Define Ragas metrics
metrics = [
    faithfulness,
    answer_relevance,
    context_relevance
]

# Run evaluation
# For Ragas to use the LLM, it needs to be configured. 
# If using OpenAI, ensure OPENAI_API_KEY is set.
# Ragas will automatically pick up the LLM from environment or can be passed explicitly.
# For MockLLM, Ragas might not work directly as it expects a specific LLM client.
# For this exercise, we assume a real LLM is used for Ragas evaluation if OPENAI_API_KEY is set.
# If only MockLLM is available, Ragas evaluation will be skipped or require a custom LLM wrapper.

# Ragas requires an LLM for its own evaluations. Let's use the same LLM if it's OpenAI.
# If it's MockLLM, we'll use a placeholder and note the limitation.
ragas_llm = None
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    ragas_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    print("Ragas will use OpenAI ChatOpenAI for its internal evaluations.")
else:
    print("Warning: Ragas requires an actual LLM for its metrics. Skipping Ragas evaluation as OPENAI_API_KEY is not set.")
    print("To run Ragas, please set your OPENAI_API_KEY environment variable.")
    ragas_llm = None # Ensure it's None if not configured

if ragas_llm:
    # Ragas needs an LLM for its internal evaluations (e.g., to judge relevance)
    # We pass the LLM explicitly to the evaluate function.
    result = evaluate(
        dataset=eval_dataset,
        metrics=metrics,
        llm=ragas_llm, # Pass the LLM for Ragas's internal use
        embeddings=None # Ragas can use embeddings for some metrics, but not strictly required for these
    )
    print("\nRagas evaluation complete.")

    # --- 4. Results Reporting ---
    print("\n--- Aggregated Ragas Scores ---")
    print(result)

    print("\n--- Detailed Ragas Scores per Sample ---")
    results_df = result.to_pandas()
    print(results_df.to_markdown(index=False))

    # --- 5. Langfuse Integration (if enabled) ---
    if langfuse_enabled:
        print("\nLogging overall evaluation results to Langfuse...")
        # Log overall scores as a Langfuse score for the entire evaluation run
        # This would typically be associated with a specific model version or experiment run.
        # For this exercise, we'll log it as a general score.
        
        # Create a new trace for the overall evaluation summary
        eval_summary_trace = langfuse.trace(name="overall_evaluation_summary", input=f"Evaluation of {len(eval_dataset_raw)} samples")
        for metric_name, score_value in result.items():
            eval_summary_trace.score(name=metric_name, value=score_value)
        eval_summary_trace.end(output=result)
        print(f"Overall evaluation results logged to Langfuse. Trace URL: {eval_summary_trace.get_trace_url()}")
        langfuse.flush()
else:
    print("Ragas evaluation skipped due to missing LLM configuration.")

print("\nExercise complete. Review the aggregated and detailed scores to understand your RAG application's performance.")
